In [4]:
%pip install ta scikit-learn pandas numpy matplotlib mplfinance torch

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.0/75.0 kB 2.7 MB/s eta 0:00:00
  Created wheel for ta: filename=ta-0.11.0-py3-none-any.whl size=29412 sha256=5f441c80a93a671edd266a167c70d631e72a2c1e3440f06877e81d08fdfb0744
  Stored in directory: /root/.cache/pip/wheels/5c/a1/5f/c6b85a7d9452057be4ce68a8e45d77ba34234a6d46581777c6
Successfully built ta


In [5]:
import pandas as pd
import numpy as np
import os
import ta
from sklearn.preprocessing import StandardScaler

# --- 1. חיבור לגוגל דרייב (מותאם לקולאב) ---
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("Google Drive mounted successfully!")
    BASE_DIR = '/content/drive/MyDrive/CryptoProject'
except ImportError:
    print("Not running in Google Colab. Using local directory.")
    try:
        BASE_DIR = os.path.dirname(os.path.abspath(__file__))
    except NameError:
        BASE_DIR = os.getcwd()

# קובץ הנתונים (5 דקות)
DATA_PATH = os.path.join(BASE_DIR, 'data', 'BTCUSDT_5m_data.csv')

# חלון זמן של יממה שלמה (24 שעות * 12 נרות בשעה)
SEQ_LENGTH = 120

PREDICT_AHEAD = 1
TRAIN_SPLIT = 0.8
VAL_SPLIT = 0.1
STEP = 5

# --- רשימת הפיצ'רים המעודכנת ---
FEATURE_COLS = [
    'log_ret', 'rsi', 'rsi_change', 'rsi_accel', 'macd', 'macd_slope',
    'bb_pband_change', 'volume', 'ma_dist', 'volume_z', 'vol_spike', 'adx',
    'hour_sin', 'hour_cos'
]

# המטרה עברה להיות Bollinger %B (כמו ב-LSTM שמצליח יותר)
TARGET_COL = 'bb_pband'

class TransformerDataPreprocessor:
    def __init__(self):
        pass

    def check_feature_correlations(self, df, feature_cols):
        """
        Check and print correlations between features to identify redundancy.
        """
        print("\n--- Feature Correlations ---")
        corr_matrix = df[feature_cols].corr()
        print(corr_matrix)

        print("\nHigh correlations (>0.8 or <-0.8) may indicate redundancy:")
        for i in range(len(feature_cols)):
            for j in range(i+1, len(feature_cols)):
                corr_val = corr_matrix.iloc[i, j]
                if abs(corr_val) > 0.8:
                    print(f"  {feature_cols[i]} vs {feature_cols[j]}: {corr_val:.3f}")

    def load_and_clean_data(self, filepath):
        print(f"Loading data from {filepath}...")

        if not os.path.exists(filepath):
            raise FileNotFoundError(f"File not found: {filepath}\nPlease make sure the API script finished running and saved the CSV to Drive.")

        df = pd.read_csv(filepath)

        df['open_time'] = pd.to_datetime(df['open_time'])
        df = df.sort_values('open_time')
        df = df.drop_duplicates(subset=['open_time'])
        df = df.ffill().dropna()

        # --- 1. חישוב Log Returns ---
        df['log_ret'] = np.log((df['close'] / df['close'].shift(1))* 10)

        # --- 2. חישוב אינדיקטורים טכניים ---
        df['rsi'] = ta.momentum.rsi(df['close'], window=14) / 100.0
        df['rsi_change'] = df['rsi'].diff(periods=3)
        df['rsi_accel'] = df['rsi_change'].diff(periods=2)

        macd = ta.trend.MACD(df['close'])
        macd_raw = macd.macd_diff()
        df['macd'] = (macd_raw - macd_raw.rolling(window=100).mean()) / (macd_raw.rolling(window=100).std() + 1e-9)
        df['macd_diff'] = macd.macd_diff()
        df['macd_slope'] = df['macd_diff'].diff(periods=2)

        bb = ta.volatility.BollingerBands(df['close'], window=20, window_dev=2)
        df['bb_pband'] = bb.bollinger_pband()
        df['bb_pband_change'] = df['bb_pband'].diff(periods=1)

        df['log_ret'] = np.log(df['close'] / df['close'].shift(1))
        df['log_ret_lag'] = df['log_ret'].shift(1)
        df['log_ret_lag_2'] = df['log_ret'].shift(2)

        # --- 3. טיפול ב-Volume ---
        df['volume'] = np.log(df['volume'] + 1)
        df['volume'] = (df['volume'] - df['volume'].mean()) / (df['volume'].std() + 1e-9)
        df['vol_ma'] = df['volume'].rolling(window=20).mean()
        df['vol_std'] = df['volume'].rolling(window=20).std()

        df['volume_z'] = (df['volume'] - df['vol_ma']) / (df['vol_std'] + 1e-9)
        df['vol_spike'] = (df['volume'] > (df['vol_ma'] * 2)).astype(float)

        # --- חישוב מרחק מהממוצע ---
        df['ma_20'] = df['close'].rolling(window=20).mean()
        df['ma_dist'] = (df['close'] - df['ma_20']) / (df['ma_20'] + 1e-9)
        df['ma_dist'] = df['ma_dist'] * 10.0

        # --- הוספת אינדיקטור עוצמת טרנד (ADX) ---
        adx = ta.trend.ADXIndicator(df['high'], df['low'], df['close'], window=14)
        df['adx'] = adx.adx()

        # --- תוספת: זמן מחזורי ל-TFT ---
        df['hour'] = df['open_time'].dt.hour
        df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
        df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)

        df = df.dropna()
        print(f"Features calculated. Total Rows available: {len(df)}")
        return df

    def create_sequences(self, data, target, seq_length, step=1):
        xs, ys = [], []
        for i in range(0, len(data) - seq_length - PREDICT_AHEAD + 1, step):
            x = data[i : (i + seq_length)]
            y = target[i + seq_length + PREDICT_AHEAD - 1]
            xs.append(x)
            ys.append(y)
        return np.array(xs), np.array(ys)

    def process(self):
        try:
            df = self.load_and_clean_data(DATA_PATH)
        except FileNotFoundError as e:
            print(e)
            return

        self.check_feature_correlations(df, FEATURE_COLS)

        data = df[FEATURE_COLS].values
        target = df[TARGET_COL].values

        n = len(data)
        train_end = int(n * TRAIN_SPLIT)
        val_end = int(n * (TRAIN_SPLIT + VAL_SPLIT))

        train_data = data[:train_end]
        train_target = target[:train_end]

        val_data = data[train_end:val_end]
        val_target = target[train_end:val_end]

        test_data = data[val_end:]
        test_target = target[val_end:]

        # --- נרמול ---
        scaler = StandardScaler()
        train_data_scaled = scaler.fit_transform(train_data)
        val_data_scaled = scaler.transform(val_data)
        test_data_scaled = scaler.transform(test_data)

        # --- יצירת חלונות ---
        print(f"\nCreating sliding windows (SEQ_LENGTH={SEQ_LENGTH}, Features={len(FEATURE_COLS)}, Step={STEP})...")
        X_train, y_train = self.create_sequences(train_data_scaled, train_target, SEQ_LENGTH, STEP)
        X_val, y_val = self.create_sequences(val_data_scaled, val_target, SEQ_LENGTH, STEP)
        X_test, y_test = self.create_sequences(test_data_scaled, test_target, SEQ_LENGTH, STEP)

        # --- שמירה לגוגל דרייב (ספרייה ייעודית לטרנספורמר) ---
        save_dir = os.path.join(BASE_DIR, 'processed_data_transformer')
        if not os.path.exists(save_dir):
            os.makedirs(save_dir)
            print(f"Created processed_data_transformer directory at: {save_dir}")

        np.save(os.path.join(save_dir, 'X_train.npy'), X_train)
        np.save(os.path.join(save_dir, 'y_train.npy'), y_train)
        np.save(os.path.join(save_dir, 'X_val.npy'), X_val)
        np.save(os.path.join(save_dir, 'y_val.npy'), y_val)
        np.save(os.path.join(save_dir, 'X_test.npy'), X_test)
        np.save(os.path.join(save_dir, 'y_test.npy'), y_test)

        print("\n--- Preprocessing Complete ---")
        print(f"Files successfully saved to: {save_dir}")
        print(f"X_train shape: {X_train.shape}")
        print(f"X_test shape: {X_test.shape}")

if __name__ == "__main__":
    processor = TransformerDataPreprocessor()
    processor.process()


Mounted at /content/drive
Google Drive mounted successfully!
Loading data from /content/drive/MyDrive/CryptoProject/data/BTCUSDT_5m_data.csv...
Features calculated. Total Rows available: 896705

--- Feature Correlations ---
                  log_ret       rsi  rsi_change  rsi_accel      macd  \
log_ret          1.000000  0.299354    0.434900   0.352444  0.160174   
rsi              0.299354  1.000000    0.346462   0.059351  0.579726   
rsi_change       0.434900  0.346462    1.000000   0.620648  0.305686   
rsi_accel        0.352444  0.059351    0.620648   1.000000 -0.135605   
macd             0.160174  0.579726    0.305686  -0.135605  1.000000   
macd_slope       0.285467  0.162515    0.586743   0.241072  0.228016   
bb_pband_change  0.646214  0.149640    0.459848   0.453248  0.005443   
volume          -0.009620 -0.018499   -0.007992  -0.003505 -0.009545   
ma_dist          0.379430  0.744173    0.263605   0.016576  0.609725   
volume_z        -0.020087 -0.018900   -0.019924  -0.0088